# Mental Diary Model Training (Colab)

Этот ноутбук умеет:
- загружать существующую модель из чекпоинта
- создавать новую модель, если чекпоинта нет
- обучать модель на `jsonl`-датасете
- сохранять чекпоинт после каждой успешной эпохи
- сохранять лучший чекпоинт отдельно


In [ ]:
!pip -q install transformers datasets scikit-learn sentencepiece accelerate

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json
import os
from pathlib import Path
from dataclasses import dataclass

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
LABELS = ['stable', 'fatigued', 'stressed', 'burnout-risk', 'recovery']
LABEL_TO_ID = {label: idx for idx, label in enumerate(LABELS)}
ID_TO_LABEL = {idx: label for label, idx in LABEL_TO_ID.items()}

@dataclass
class Config:
    dataset_path: str = '/content/drive/MyDrive/mental-diary/training_samples.jsonl'
    checkpoint_dir: str = '/content/drive/MyDrive/mental-diary/checkpoints'
    model_name: str = 'xlm-roberta-base'
    max_length: int = 256
    batch_size: int = 8
    learning_rate: float = 2e-5
    epochs: int = 5
    weight_decay: float = 0.01
    val_size: float = 0.2
    random_state: int = 17
    gradient_clip: float = 1.0

CFG = Config()
Path(CFG.checkpoint_dir).mkdir(parents=True, exist_ok=True)
print('Device:', DEVICE)
print(CFG)

In [ ]:
def load_jsonl(path: str):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

samples = load_jsonl(CFG.dataset_path)
texts = [sample['text'] for sample in samples]
labels = [LABEL_TO_ID[sample['label']] for sample in samples]

train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts,
    labels,
    test_size=CFG.val_size,
    random_state=CFG.random_state,
    stratify=labels,
)

print('Train:', len(train_texts), 'Val:', len(val_texts))

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)

class MentalDiaryDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoded['input_ids'].squeeze(0),
            'attention_mask': encoded['attention_mask'].squeeze(0),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long),
        }

class MentalStateClassifier(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        logits = self.classifier(self.dropout(pooled))
        return logits

train_ds = MentalDiaryDataset(train_texts, train_labels, tokenizer, CFG.max_length)
val_ds = MentalDiaryDataset(val_texts, val_labels, tokenizer, CFG.max_length)
train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=CFG.batch_size, shuffle=False)

In [ ]:
def checkpoint_path(name: str) -> str:
    return str(Path(CFG.checkpoint_dir) / name)

def save_checkpoint(path, model, optimizer, scheduler, epoch, best_f1):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'best_f1': best_f1,
        'config': CFG.__dict__,
        'labels': LABELS,
    }, path)

def load_or_create_model(resume_path=None):
    model = MentalStateClassifier(CFG.model_name, len(LABELS)).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.learning_rate, weight_decay=CFG.weight_decay)
    total_steps = len(train_loader) * CFG.epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=max(1, total_steps // 10),
        num_training_steps=total_steps,
    )
    start_epoch = 0
    best_f1 = 0.0

    if resume_path and os.path.exists(resume_path):
        checkpoint = torch.load(resume_path, map_location=DEVICE)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        if checkpoint.get('scheduler_state_dict') is not None:
            scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_f1 = checkpoint.get('best_f1', 0.0)
        print(f'Resumed from {resume_path}, epoch={start_epoch}, best_f1={best_f1:.4f}')
    else:
        print('No checkpoint found, creating a new model')

    return model, optimizer, scheduler, start_epoch, best_f1

RESUME_CHECKPOINT = checkpoint_path('latest.pt')
model, optimizer, scheduler, start_epoch, best_f1 = load_or_create_model(RESUME_CHECKPOINT)

In [ ]:
def evaluate(model, loader):
    model.eval()
    all_preds = []
    all_labels = []
    total_loss = 0.0
    loss_fn = nn.CrossEntropyLoss()

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)

            logits = model(input_ids, attention_mask)
            loss = loss_fn(logits, labels)
            total_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    metrics = {
        'loss': total_loss / max(1, len(loader)),
        'accuracy': accuracy_score(all_labels, all_preds),
        'macro_f1': f1_score(all_labels, all_preds, average='macro'),
        'report': classification_report(all_labels, all_preds, target_names=LABELS, zero_division=0)
    }
    return metrics

def train_one_epoch(model, loader, optimizer, scheduler):
    model.train()
    loss_fn = nn.CrossEntropyLoss()
    total_loss = 0.0

    for batch in loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)

        logits = model(input_ids, attention_mask)
        loss = loss_fn(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.gradient_clip)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    return total_loss / max(1, len(loader))

for epoch in range(start_epoch, CFG.epochs):
    print(f'\nEpoch {epoch + 1}/{CFG.epochs}')
    train_loss = train_one_epoch(model, train_loader, optimizer, scheduler)
    metrics = evaluate(model, val_loader)
    print({'train_loss': train_loss, 'val_loss': metrics['loss'], 'accuracy': metrics['accuracy'], 'macro_f1': metrics['macro_f1']})
    print(metrics['report'])

    latest_path = checkpoint_path('latest.pt')
    save_checkpoint(latest_path, model, optimizer, scheduler, epoch, best_f1)
    print(f'Saved latest checkpoint to {latest_path}')

    if metrics['macro_f1'] > best_f1:
        best_f1 = metrics['macro_f1']
        best_path = checkpoint_path('best.pt')
        save_checkpoint(best_path, model, optimizer, scheduler, epoch, best_f1)
        print(f'New best model saved to {best_path}')


In [ ]:
def predict_text(text, checkpoint=None):
    local_model, _, _, _, _ = load_or_create_model(checkpoint or checkpoint_path('best.pt'))
    local_model.eval()
    encoded = tokenizer(text, truncation=True, padding='max_length', max_length=CFG.max_length, return_tensors='pt')
    with torch.no_grad():
        logits = local_model(
            encoded['input_ids'].to(DEVICE),
            encoded['attention_mask'].to(DEVICE)
        )
        probs = torch.softmax(logits, dim=1).squeeze(0).cpu().tolist()
    result = {ID_TO_LABEL[idx]: round(prob, 4) for idx, prob in enumerate(probs)}
    return result

predict_text('Мне пусто и безнадёжно, ничего не радует, не хочу ни с кем говорить.')